# 03 Aggregate-Only Modeling: Predicting LoL Match Result From 10-Minute State

Notebook นี้เป็น modeling notebook แบบ **aggregate-only** สำหรับใช้บน Kaggle โดยใช้ข้อมูลสถิติช่วง 10 นาทีแรกของเกมเพื่อทำนายว่า Blue Team จะชนะหรือไม่

บทบาทของ notebook นี้คือเป็นโมเดล baseline ที่ตีความง่าย: ใช้ team-level features และ difference features ที่สอดคล้องกับ insight จาก EDA เช่น gold difference, KDA difference, objective difference และ damage difference


## EDA-Driven Modeling Rationale

จาก `02_EDA.ipynb` เราเห็นว่า feature ที่เกี่ยวกับ **economy**, **combat**, **damage** และ **map objectives** ในช่วง 10 นาทีแรกสัมพันธ์กับผลแพ้ชนะอย่างชัดเจน โดยเฉพาะ:

- **Gold difference** แยกเกมที่ Blue ชนะ/แพ้ได้ดี และ win probability เพิ่มขึ้นตาม gold lead
- **Lane matchup difference** ให้ข้อมูลดีกว่าค่าดิบของฝั่ง Blue อย่างเดียว เพราะ LoL เป็นเกมที่ต้องเทียบความได้เปรียบกับคู่แข่งโดยตรง
- **Kill/KDA difference** และ **damage difference** สะท้อนการชนะไฟต์และแรงกดดันช่วงต้นเกม
- **Objective control** เช่น Dragon, Rift Herald/Voidgrubs และ Tower ช่วยบอกการควบคุมแผนที่

ดังนั้น feature engineering ใน notebook นี้จึงเน้น `Diff_*`, `Blue_*`, และ `Red_*` เพื่อให้โมเดลเรียนรู้ “ทีมไหนนำอยู่ในนาทีที่ 10” แทนการใช้ raw player columns ทั้งหมด

สำคัญ: ข้อมูลนี้เป็น **10-minute game state** ไม่ใช่ post-game statistics จึงไม่ได้เป็นการให้โมเดลเห็นผลลัพธ์หลังจบเกมโดยตรง


In [ ]:
import sys
import os
os.environ["PYTHONIOENCODING"] = "utf-8"
try:
    sys.stdout.reconfigure(encoding="utf-8")
except Exception:
    pass

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import joblib
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, roc_auc_score, f1_score,
    classification_report, confusion_matrix, RocCurveDisplay, 
    ConfusionMatrixDisplay
)

import xgboost as xgb
import lightgbm as lgb
import catboost as cb
from catboost import CatBoostClassifier


## 1. Load Data

อ่านไฟล์ `merged_v1.csv` ที่สร้างจาก notebook ขั้น data merging โดยหนึ่งแถวแทนหนึ่ง match และมี target เป็น `BlueWin` / `RedWin` ก่อนเริ่ม model เราตรวจ shape เพื่อยืนยันจำนวนแถวและจำนวน column ที่จะเข้าสู่ pipeline


In [ ]:
DATA_PATH = Path("/kaggle/input/datasets/kiatisakkk/cpe232-lol-10min-dataset/t2_transformed/merged_v1.csv")

In [ ]:
df = pd.read_csv(DATA_PATH)
df.shape

## 2. Basic Cleaning

ตัด column ที่เป็น identifier หรือข้อมูลที่ไม่ควรให้โมเดลเรียนโดยตรง เช่น `MatchFk` และ `Patch` เพราะไม่ได้เป็น signal เชิงพฤติกรรมของเกม จากนั้นลบ match ที่ `BlueWin` และ `RedWin` ขัดแย้งกันหรือมีค่าเท่ากัน เพื่อให้ target เป็น binary label ที่ชัดเจน แล้ว drop `RedWin` เหลือ target เดียวคือ `BlueWin`

ค่า lane ที่เป็น `NONE` ถูก map เป็น `JUNGLE` เพราะในบริบทเกม LoL มักหมายถึง role jungle และช่วยลด category ที่ไม่สื่อความหมาย


In [ ]:
drop_id_cols = ["MatchFk", "Patch"]
df = df.drop(columns=[c for c in drop_id_cols if c in df.columns], errors="ignore")

In [ ]:
df = df[df["BlueWin"] != df["RedWin"]].reset_index(drop=True)
df = df.drop(columns=["RedWin"], errors="ignore")

## 2.1 Target And Queue Distribution

ก่อนสร้าง feature ตรวจ target balance เพื่อดูว่า Accuracy อ่านได้ตรงไปตรงมาหรือไม่ และตรวจ `QueueType` เพื่อยืนยันว่า notebook กำลัง model เกมประเภทใดบ้าง


In [ ]:
target_balance = (
    df["BlueWin"]
    .value_counts()
    .rename_axis("BlueWin")
    .reset_index(name="count")
)
target_balance["ratio"] = target_balance["count"] / target_balance["count"].sum()

display(target_balance)

if "QueueType" in df.columns:
    queue_summary = (
        df["QueueType"]
        .value_counts()
        .rename_axis("QueueType")
        .reset_index(name="count")
    )
    queue_summary["ratio"] = queue_summary["count"] / queue_summary["count"].sum()
    display(queue_summary)


In [ ]:
lane_cols = [f"Lane_P{p}" for p in range(1, 11)]
for c in lane_cols:
    df[c] = df[c].replace("NONE", "JUNGLE")

## 3. Encode Categorical Features

Column แบบข้อความ เช่น lane หรือ category อื่น ๆ ถูกแปลงด้วย one-hot encoding เพื่อให้โมเดล tree-based, linear model และ neural network ใช้งานได้โดยไม่สร้างลำดับเทียมระหว่าง category


In [ ]:
ohe_cols = df.select_dtypes(include="object").columns.tolist()

df = pd.get_dummies(df, columns=ohe_cols)

## 4. Feature Engineering

สร้าง feature ระดับทีมจากข้อมูลผู้เล่น 5 คนต่อฝั่ง เช่น gold, minion, kills, deaths, assists และ damage โดยคำนวณทั้งผลรวม ค่าเฉลี่ย และผลต่างระหว่าง Blue กับ Red

แนวคิดสำคัญคือโมเดลควรเห็น “ความได้เปรียบของทีม” มากกว่าค่าดิบของผู้เล่นแยกคนอย่างเดียว เช่น `Diff_TotalGold`, `Diff_KDA`, `Diff_DragonKills` และ `Diff_TowerKills` มักสัมพันธ์กับโอกาสชนะของทีมโดยตรง


In [ ]:
# --- Team-level aggregates ---
blue_players = [f"P{i}" for i in range(1, 6)]
red_players  = [f"P{i}" for i in range(6, 11)]

for stat in ["TotalGold", "MinionsKilled", "kills", "deaths", "assists", "DmgDealt"]:
    blue_cols = [f"{stat}_{p}" for p in blue_players]
    red_cols  = [f"{stat}_{p}" for p in red_players]
    df[f"Blue_{stat}_sum"] = df[blue_cols].sum(axis=1)
    df[f"Red_{stat}_sum"]  = df[red_cols].sum(axis=1)
    df[f"Blue_{stat}_avg"] = df[blue_cols].mean(axis=1)
    df[f"Red_{stat}_avg"]  = df[red_cols].mean(axis=1)
    df[f"Diff_{stat}"]     = df[f"Blue_{stat}_sum"] - df[f"Red_{stat}_sum"]

# --- KDA ---
def safe_kda(k, d, a):
    return (k + a) / d.replace(0, 1)

df["Blue_KDA"] = safe_kda(df["Blue_kills_sum"], df["Blue_deaths_sum"], df["Blue_assists_sum"])
df["Red_KDA"]  = safe_kda(df["Red_kills_sum"],  df["Red_deaths_sum"],  df["Red_assists_sum"])
df["Diff_KDA"] = df["Blue_KDA"] - df["Red_KDA"]

# --- Objective differentials (already team-level) ---
for obj in ["BaronKills", "RiftHeraldKills", "DragonKills", "TowerKills"]:
    df[f"Diff_{obj}"] = df[f"Blue{obj}"] - df[f"Red{obj}"]

# Kill differential (already team-level)
df["Diff_Kills"] = df["BlueKills"] - df["RedKills"]

# --- Gold share (how evenly gold is distributed) ---
for side, players in [("Blue", blue_players), ("Red", red_players)]:
    gold_cols = [f"TotalGold_{p}" for p in players]
    df[f"{side}_GoldStd"] = df[gold_cols].std(axis=1)

## 5. Select Aggregate Feature Set

ใน notebook นี้เลือกใช้ strategy แบบ **aggregate-only** คือใช้ feature ระดับทีมและ feature ผลต่างเป็นหลัก เพื่อลด noise จาก column รายผู้เล่นจำนวนมาก และทำให้โมเดลตีความง่ายขึ้นเมื่อดู feature importance ภายหลัง


In [ ]:
# Strategy A — "Aggregate-only" (team-level features, no per-player columns)
agg_feature_cols = [c for c in df.columns if c.startswith(("Blue_", "Red_", "Diff_"))]

# Include objective raw counts because EDA shows that map control matters at 10 minutes.
raw_objective_cols = [
    "BlueBaronKills", "BlueRiftHeraldKills", "BlueDragonKills", "BlueTowerKills", "BlueKills",
    "RedBaronKills", "RedRiftHeraldKills", "RedDragonKills", "RedTowerKills", "RedKills",
]
agg_feature_cols += [c for c in raw_objective_cols if c in df.columns]

# Deduplicate and keep only existing columns.
agg_feature_cols = sorted(set(c for c in agg_feature_cols if c in df.columns))

def aggregate_feature_group(col):
    if col.startswith("Diff_"):
        return "difference_features"
    if col in raw_objective_cols:
        return "raw_objective_counts"
    if col.startswith(("Blue_", "Red_")):
        return "team_aggregate_features"
    return "other"

feature_group_summary = (
    pd.Series([aggregate_feature_group(c) for c in agg_feature_cols])
    .value_counts()
    .rename_axis("feature_group")
    .reset_index(name="feature_count")
)

feature_set_summary = pd.DataFrame([
    {"item": "Rows", "value": len(df)},
    {"item": "Aggregate-only feature count", "value": len(agg_feature_cols)},
    {"item": "Blue win ratio", "value": df["BlueWin"].mean()},
    {"item": "Primary metric", "value": "AUC-ROC"},
])

display(feature_set_summary)
display(feature_group_summary)


## 6. Train / Validation / Test Split

แบ่งข้อมูลเป็น train, validation และ test แบบ stratified เพื่อรักษาสัดส่วน win/loss ในแต่ละชุด

- **Train set** ใช้ fit โมเดลระหว่าง tuning
- **Validation set** ใช้เลือก hyperparameter ที่ดีที่สุด
- **Test set** เก็บไว้ประเมินครั้งสุดท้าย เพื่อประมาณ performance กับข้อมูลที่โมเดลไม่เคยเห็น

สร้างข้อมูลสองชุดคือแบบ scaled และ unscaled เพราะ Logistic Regression กับ MLP ไวต่อ scale ของ feature ส่วน tree-based models ไม่จำเป็นต้อง standardize


In [ ]:
def prepare_data(df, feature_cols=None, target="BlueWin", val_size=0.15, test_size=0.15, random_state=1337, scale=False):

    if feature_cols is None:
        feature_cols = [c for c in df.columns if c != target]

    X = df[feature_cols].copy()
    y = df[target].copy()

    temp_size = val_size + test_size
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=temp_size, random_state=random_state, stratify=y
    )

    relative_test_size = test_size / temp_size
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=relative_test_size, random_state=random_state, stratify=y_temp
    )

    scaler = None
    X_train_sc, X_val_sc, X_test_sc = X_train, X_val, X_test

    if scale:
        scaler = StandardScaler()
        X_train_sc = scaler.fit_transform(X_train)
        X_val_sc   = scaler.transform(X_val)
        X_test_sc  = scaler.transform(X_test)

    return {
        "X_train": X_train_sc,
        "X_val":   X_val_sc,
        "X_test":  X_test_sc,
        "y_train": y_train,
        "y_val":   y_val,
        "y_test":  y_test,
        "scaler":  scaler
    }

def unpack_data(data):
    return (
        data["X_train"],
        data["X_val"],
        data["X_test"],
        data["y_train"],
        data["y_val"],
        data["y_test"],
    )

In [ ]:
data_sc = prepare_data(df, feature_cols=agg_feature_cols, scale=True)
X_train_sc, X_val_sc, X_test_sc, y_train, y_val, y_test = unpack_data(data_sc)

data = prepare_data(df, feature_cols=agg_feature_cols)
X_train, X_val, X_test, y_train, y_val, y_test = unpack_data(data)

## 7. Evaluation Helper

ฟังก์ชันนี้เป็น helper สำหรับ train และวัดผลโมเดลด้วย Accuracy, AUC และ F1 โดยใช้ `predict_proba` สำหรับคำนวณ AUC ทำให้ผลลัพธ์ของแต่ละ model ถูกเก็บใน format เดียวกัน


In [ ]:
def train_and_evaluate(name, model, data, results=None):
    if results is None:
        results = {}

    X_train, X_val, X_test, y_train, y_val, y_test = unpack_data(data)

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    f1  = f1_score(y_test, y_pred)

    results[name] = {
        "model":  model,
        "y_pred": y_pred,
        "y_prob": y_prob,
        "acc":    acc,
        "auc":    auc,
        "f1":     f1,
    }

    print(f" > {name}")
    print(f"   Accuracy: {acc:.4f}  |  AUC: {auc:.4f}  |  F1: {f1:.4f}")

    return results

## 8. Hyperparameter Tuning With Optuna

ใช้ Optuna ทำ hyperparameter tuning โดย objective ของทุกโมเดลคือ maximize **Validation AUC** ซึ่งเหมาะกับ binary classification และไม่ผูกกับ threshold เดียว

Logistic Regression ถูกเก็บไว้เป็น baseline สำคัญ แม้มันอาจทำคะแนนสูง เพราะ aggregate difference features เช่น `Diff_TotalGold`, `Diff_KDA`, `Diff_DmgDealt` และ `Diff_DragonKills` มีความสัมพันธ์ค่อนข้างตรงกับผลแพ้ชนะ ทำให้ linear decision boundary อาจเพียงพอ

Tree-based models ใช้เพื่อทดสอบว่า nonlinear interactions เพิ่มประสิทธิภาพจาก aggregate-only features ได้มากแค่ไหน ส่วน XGBoost, LightGBM และ CatBoost จะพยายามใช้ GPU บน Kaggle และ fallback เป็น CPU ถ้า environment ไม่รองรับ


In [ ]:
N_TRIALS = 50
USE_GPU = True
GPU_FALLBACK_TO_CPU = True

def gpu_params_for(model_name):
    if not USE_GPU:
        return {}

    if model_name == "XGBoost":
        return {
            "tree_method": "hist",
            "device": "cuda",
        }

    if model_name == "LightGBM":
        return {
            "device_type": "gpu",
        }

    if model_name == "CatBoost":
        return {
            "task_type": "GPU",
            "devices": "0",
        }

    return {}


def fit_with_optional_gpu_fallback(model_name, build_model, fit_fn):
    model = build_model(use_gpu=USE_GPU)

    try:
        fit_fn(model)
        return model

    except Exception as exc:
        can_fallback = (
            USE_GPU
            and GPU_FALLBACK_TO_CPU
            and model_name in {"XGBoost", "LightGBM", "CatBoost"}
        )

        if not can_fallback:
            raise

        print(f"GPU failed for {model_name}; retrying on CPU.")
        print(f"Reason: {type(exc).__name__}: {exc}")

        model = build_model(use_gpu=False)
        fit_fn(model)
        return model


def obj_logreg(trial):
    m = LogisticRegression(
        C=1.0,
        solver="lbfgs",
        penalty="l2",
        max_iter=500,
        random_state=42,
    )

    m.fit(X_train_sc, y_train)
    return roc_auc_score(y_val, m.predict_proba(X_val_sc)[:, 1])


def obj_dt(trial):
    m = DecisionTreeClassifier(
        max_depth=trial.suggest_int("max_depth", 2, 20),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 30),
        max_features=trial.suggest_categorical(
            "max_features",
            ["sqrt", "log2", None],
        ),
        random_state=42,
    )

    m.fit(X_train.values, y_train.values)
    return roc_auc_score(y_val, m.predict_proba(X_val.values)[:, 1])


def obj_rf(trial):
    m = RandomForestClassifier(
        n_estimators=trial.suggest_int("n_estimators", 100, 300, step=50),
        max_depth=trial.suggest_int("max_depth", 6, 16),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 2, 20),
        max_features=trial.suggest_categorical(
            "max_features",
            ["sqrt", "log2"],
        ),
        bootstrap=True,
        max_samples=trial.suggest_categorical(
            "max_samples",
            [0.5, 0.75],
        ),
        random_state=42,
        n_jobs=-1,
    )

    m.fit(X_train.values, y_train.values)
    return roc_auc_score(y_val, m.predict_proba(X_val.values)[:, 1])


def obj_xgb(trial):
    params = {
        "n_estimators": 800,
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "subsample": trial.suggest_float("subsample", 0.7, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 15),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10, log=True),
        "eval_metric": "logloss",
        "early_stopping_rounds": 50,
        "random_state": 42,
        "n_jobs": -1,
        "verbosity": 0,
    }

    def build_model(use_gpu):
        if use_gpu:
            gpu_params = {
                **gpu_params_for("XGBoost"),
                "sampling_method": "gradient_based",
            }
        else:
            gpu_params = {
                "tree_method": "hist",
                "sampling_method": "uniform",
            }

        return xgb.XGBClassifier(**params, **gpu_params)

    m = fit_with_optional_gpu_fallback(
        "XGBoost",
        build_model,
        lambda model: model.fit(
            X_train.values,
            y_train.values,
            eval_set=[(X_val.values, y_val.values)],
            verbose=False,
        ),
    )

    return roc_auc_score(y_val, m.predict_proba(X_val.values)[:, 1])



def obj_lgb(trial):
    params = {
        "n_estimators": 800,
        "num_leaves": trial.suggest_int("num_leaves", 16, 128),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "subsample": trial.suggest_float("subsample", 0.7, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 80),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10, log=True),
        "random_state": 42,
        "n_jobs": -1,
        "verbose": -1,
    }

    def build_model(use_gpu):
        gpu_params = gpu_params_for("LightGBM") if use_gpu else {}
        return lgb.LGBMClassifier(**params, **gpu_params)

    m = fit_with_optional_gpu_fallback(
        "LightGBM",
        build_model,
        lambda model: model.fit(
            X_train.values,
            y_train.values,
            eval_set=[(X_val.values, y_val.values)],
            callbacks=[lgb.early_stopping(50, verbose=False)],
        ),
    )

    return roc_auc_score(y_val, m.predict_proba(X_val.values)[:, 1])


def obj_catboost(trial):
    params = {
        "iterations": 800,
        "depth": trial.suggest_int("depth", 4, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-2, 10, log=True),
        "random_strength": trial.suggest_float("random_strength", 0.1, 5, log=True),
        "random_seed": 42,
        "verbose": 0,
        "early_stopping_rounds": 50,
    }

    def build_model(use_gpu):
        gpu_params = gpu_params_for("CatBoost") if use_gpu else {}
        return CatBoostClassifier(**params, **gpu_params)

    m = fit_with_optional_gpu_fallback(
        "CatBoost",
        build_model,
        lambda model: model.fit(
            X_train.values,
            y_train.values,
            eval_set=(X_val.values, y_val.values),
        ),
    )

    return roc_auc_score(y_val, m.predict_proba(X_val.values)[:, 1])


## 9. Run Tuning

กำหนด `N_TRIALS = 50` เพื่อให้ Optuna ทดลองค่าพารามิเตอร์หลายชุดต่อโมเดล จากนั้นเก็บ `study.best_params` และ `study.best_value` ไว้ใช้ตอน train model ใหม่

ถ้าต้องการรันเร็วขึ้นระหว่างทดลอง สามารถลด `N_TRIALS` ได้ เช่น 10-20 trials แล้วค่อยเพิ่มเมื่อจะทำ final run โดยรอบนี้ตัด `GradientBoosting` และ `MLP` ออกเพื่อให้ tuning เร็วขึ้นและโฟกัสกับโมเดลหลัก


In [ ]:
studies = {}
objectives = {
    "LogisticRegression": obj_logreg,
    "DecisionTree":       obj_dt,
    "RandomForest":       obj_rf,
    "XGBoost":            obj_xgb,
    "LightGBM":           obj_lgb,
    "CatBoost":           obj_catboost,
}

MODEL_TRIALS = {
    "LogisticRegression": 1,
    "DecisionTree": 100,
    "RandomForest": 100,
    "XGBoost": 100,
    "LightGBM": 100,
    "CatBoost": 100,
}

for name, obj_fn in objectives.items():
    n_trials = MODEL_TRIALS.get(name, N_TRIALS)

    print(f"\n  > Tuning {name} ({n_trials} trials) ...")
    study = optuna.create_study(direction="maximize", study_name=name)
    study.optimize(obj_fn, n_trials=n_trials, show_progress_bar=False)
    studies[name] = study

    print(f"Best Val AUC: {study.best_value:.4f}")
    print(f"Best Params:  {study.best_params}")



## 10. Retrain Tuned Models And Evaluate Test Set

หลัง tuning เสร็จ เราสร้างโมเดลใหม่จาก `best_params` ของแต่ละ Optuna study แล้ว train อีกครั้งด้วย train set เดิม จากนั้น evaluate บน test set ที่ยังไม่ถูกใช้ใน tuning

ขั้นนี้ช่วยเปรียบเทียบว่าโมเดลที่ tune แล้วแต่ละตัวทำงานบน unseen data ได้ดีแค่ไหน โดยยังใช้ validation set เฉพาะกับโมเดล boosting ที่ต้องการ early stopping


In [ ]:
def build_best_model(name, params, use_gpu=USE_GPU):
    if name == "LogisticRegression":
        return LogisticRegression(C=1.0, solver="lbfgs", penalty="l2", max_iter=500, random_state=42)
    if name == "DecisionTree":
        return DecisionTreeClassifier(**params, random_state=42)
    if name == "RandomForest":
        return RandomForestClassifier(**params, bootstrap=True, random_state=42, n_jobs=-1)
    if name == "XGBoost":
        if use_gpu:
            gpu_params = {**gpu_params_for("XGBoost"), "sampling_method": "gradient_based"}
        else:
            gpu_params = {"tree_method": "hist", "sampling_method": "uniform"}
        return xgb.XGBClassifier(
            **params,
            n_estimators=800,
            eval_metric="logloss",
            early_stopping_rounds=50,
            random_state=42,
            n_jobs=-1,
            verbosity=0,
            **gpu_params,
        )
    if name == "LightGBM":
        gpu_params = gpu_params_for("LightGBM") if use_gpu else {}
        return lgb.LGBMClassifier(
            **params,
            n_estimators=800,
            random_state=42,
            n_jobs=-1,
            verbose=-1,
            **gpu_params,
        )
    if name == "CatBoost":
        gpu_params = gpu_params_for("CatBoost") if use_gpu else {}
        return CatBoostClassifier(
            **params,
            iterations=800,
            random_seed=42,
            verbose=0,
            early_stopping_rounds=50,
            **gpu_params,
        )
    raise ValueError(f"Unknown model name: {name}")

results = {}

for name, study in studies.items():
    bp = study.best_params.copy()
    uses_scaled = name == "LogisticRegression"
    Xtr = X_train_sc if uses_scaled else X_train.values
    Xv = X_val_sc if uses_scaled else X_val.values
    Xte = X_test_sc if uses_scaled else X_test.values

    def fit_model(model):
        if name == "XGBoost":
            model.fit(Xtr, y_train.values, eval_set=[(Xv, y_val.values)], verbose=False)
        elif name == "LightGBM":
            model.fit(
                Xtr,
                y_train.values,
                eval_set=[(Xv, y_val.values)],
                callbacks=[lgb.early_stopping(50, verbose=False)],
            )
        elif name == "CatBoost":
            model.fit(Xtr, y_train.values, eval_set=(Xv, y_val.values))
        else:
            model.fit(Xtr, y_train.values)

    model = fit_with_optional_gpu_fallback(
        name,
        lambda use_gpu: build_best_model(name, bp, use_gpu=use_gpu),
        fit_model,
    )

    y_pred = model.predict(Xte)
    y_prob = model.predict_proba(Xte)[:, 1]

    results[name] = {
        "model": model,
        "y_pred": y_pred,
        "y_prob": y_prob,
        "acc": accuracy_score(y_test, y_pred),
        "auc": roc_auc_score(y_test, y_prob),
        "f1": f1_score(y_test, y_pred),
        "best_params": study.best_params,
        "val_auc": study.best_value,
    }

    print(f"\n> {name}")
    print(f"  Val AUC:  {study.best_value:.4f}")
    print(f"  Test Acc: {results[name]['acc']:.4f} | Test AUC: {results[name]['auc']:.4f} | Test F1: {results[name]['f1']:.4f}")


## 11. Numeric Model Comparison And Feature Importance

ส่วนนี้ใช้สำหรับ presentation โดยเน้นตัวเลขก่อน:

1. เปรียบเทียบทุกโมเดลด้วย `Test_AUC`, `Test_Accuracy`, `Test_F1`
2. เลือก **Best Model** จาก `Test_AUC`
3. เทียบ **Best Model** กับ **Logistic Regression**
4. แสดง feature importance/coefficient เฉพาะสองโมเดลนี้


In [ ]:
print("\n" + "=" * 60)
print("4. NUMERIC MODEL COMPARISON")
print("=" * 60)

OUTPUT_DIR = Path("outputs/aggregate_only")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

summary = pd.DataFrame({
    name: {
        "Val_AUC": r["val_auc"],
        "Test_Accuracy": r["acc"],
        "Test_AUC": r["auc"],
        "Test_F1": r["f1"],
    }
    for name, r in results.items()
}).T.sort_values("Test_AUC", ascending=False)

summary["AUC_Rank"] = summary["Test_AUC"].rank(ascending=False, method="min").astype(int)
summary["Gap_vs_Best_AUC"] = summary["Test_AUC"].max() - summary["Test_AUC"]
summary["Gap_vs_LogReg_AUC"] = summary["Test_AUC"] - summary.loc["LogisticRegression", "Test_AUC"]

display(summary)
summary.to_csv(OUTPUT_DIR / "summary.csv")
summary.to_csv(OUTPUT_DIR / "model_numeric_comparison.csv")

params_df = pd.DataFrame({name: r["best_params"] for name, r in results.items()}).T
params_df.to_csv(OUTPUT_DIR / "best_params.csv")

best_name = summary.index[0]
best_result = results[best_name]

key_model_names = ["LogisticRegression"]
if best_name not in key_model_names:
    key_model_names.append(best_name)

key_model_summary = summary.loc[key_model_names, [
    "AUC_Rank",
    "Val_AUC",
    "Test_AUC",
    "Test_Accuracy",
    "Test_F1",
    "Gap_vs_Best_AUC",
    "Gap_vs_LogReg_AUC",
]]
display(key_model_summary)
key_model_summary.to_csv(OUTPUT_DIR / "logreg_vs_best_summary.csv")

print(f"Best model by Test AUC: {best_name}")
print(f"Logistic Regression Test AUC: {summary.loc['LogisticRegression', 'Test_AUC']:.4f}")
print(f"Best Model Test AUC: {best_result['auc']:.4f}")
print(f"AUC gap (Best - Logistic): {best_result['auc'] - summary.loc['LogisticRegression', 'Test_AUC']:+.4f}")

# --- Visual model comparison: all models, numeric metrics only ---
metrics = ["Test_AUC", "Test_Accuracy", "Test_F1"]
summary_long = (
    summary
    .reset_index()
    .rename(columns={"index": "Model"})
    .melt(
        id_vars="Model",
        value_vars=metrics,
        var_name="Metric",
        value_name="Score",
    )
)

plt.figure(figsize=(12, 6))
ax = sns.barplot(data=summary_long, x="Model", y="Score", hue="Metric")
ax.set_title("Aggregate-Only Model Comparison")
ax.set_xlabel("")
ax.set_ylabel("Score")
ax.set_ylim(0, 1)
ax.tick_params(axis="x", rotation=30)
for container in ax.containers:
    ax.bar_label(container, fmt="%.3f", fontsize=8, padding=2)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

plt.figure(figsize=(7, 4))
key_plot = key_model_summary.reset_index().rename(columns={"index": "Model"})
ax = sns.barplot(data=key_plot, x="Model", y="Test_AUC", color="#F28E2B")
ax.set_title("Logistic Regression vs Best Model")
ax.set_xlabel("")
ax.set_ylabel("Test AUC")
ax.set_ylim(0, 1)
for container in ax.containers:
    ax.bar_label(container, fmt="%.4f", fontsize=10, padding=3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "logreg_vs_best_auc.png", dpi=150, bbox_inches="tight")
plt.show()

# --- ROC curve and confusion matrix for best model ---
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
RocCurveDisplay.from_predictions(y_test, best_result["y_prob"], ax=axes[0])
axes[0].set_title(f"ROC Curve: {best_name}")

ConfusionMatrixDisplay.from_predictions(
    y_test,
    best_result["y_pred"],
    display_labels=["Red Win", "Blue Win"],
    cmap="Blues",
    ax=axes[1],
)
axes[1].set_title(f"Confusion Matrix: {best_name}")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "best_model_eval.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nClassification Report: Best Model")
print(classification_report(y_test, best_result["y_pred"], target_names=["Red Win", "Blue Win"]))

if best_name != "LogisticRegression":
    print("\nClassification Report: Logistic Regression")
    print(classification_report(y_test, results["LogisticRegression"]["y_pred"], target_names=["Red Win", "Blue Win"]))

def build_feature_signal_table(model_name, feature_cols, feature_group_fn):
    model = results[model_name]["model"]
    if hasattr(model, "coef_"):
        table = pd.DataFrame({
            "model": model_name,
            "feature": feature_cols,
            "raw_value": model.coef_[0],
        })
        table["importance"] = table["raw_value"].abs()
        table["direction"] = np.where(table["raw_value"] >= 0, "Blue-favored", "Red-favored")
        table["importance_type"] = "absolute_logistic_coefficient"
    elif hasattr(model, "feature_importances_"):
        table = pd.DataFrame({
            "model": model_name,
            "feature": feature_cols,
            "raw_value": model.feature_importances_,
            "importance": model.feature_importances_,
        })
        table["direction"] = "higher importance"
        table["importance_type"] = "tree_feature_importance"
    else:
        return None

    table["feature_group"] = table["feature"].map(feature_group_fn)
    table = table.sort_values("importance", ascending=False).reset_index(drop=True)
    table["importance_rank"] = np.arange(1, len(table) + 1)
    return table

feature_signal_tables = []
for model_name in key_model_names:
    table = build_feature_signal_table(model_name, agg_feature_cols, aggregate_feature_group)
    if table is not None:
        feature_signal_tables.append(table)

if feature_signal_tables:
    feature_importance_df = pd.concat(feature_signal_tables, ignore_index=True)
    feature_importance_df.to_csv(OUTPUT_DIR / "feature_importance.csv", index=False)

    for model_name in key_model_names:
        model_signals = feature_importance_df[feature_importance_df["model"] == model_name].head(20)
        if model_signals.empty:
            continue

        display(model_signals[[
            "model", "importance_rank", "feature", "importance", "raw_value", "direction", "feature_group"
        ]])

        plt.figure(figsize=(10, 7))
        sns.barplot(data=model_signals, x="importance", y="feature", hue="feature_group", dodge=False)
        plt.title(f"Top Feature Importance: {model_name}")
        plt.xlabel("Importance")
        plt.ylabel("")
        plt.tight_layout()
        safe_name = model_name.lower().replace(" ", "_")
        plt.savefig(OUTPUT_DIR / f"feature_importance_{safe_name}.png", dpi=150, bbox_inches="tight")
        plt.show()

    generic_model = best_name if best_name in key_model_names else key_model_names[0]
    generic_signals = feature_importance_df[feature_importance_df["model"] == generic_model].head(20)
    if not generic_signals.empty:
        plt.figure(figsize=(10, 7))
        sns.barplot(data=generic_signals, x="importance", y="feature", hue="feature_group", dodge=False)
        plt.title(f"Top Feature Importance: {generic_model}")
        plt.xlabel("Importance")
        plt.ylabel("")
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / "feature_importance.png", dpi=150, bbox_inches="tight")
        plt.show()
else:
    print("Selected models do not expose coefficients or feature_importances_.")


## 12. Train Final Model ใหม่ด้วย Train + Validation

เมื่อเลือกโมเดลที่ดีที่สุดแล้ว ขั้นสุดท้ายคือสร้าง final model ใหม่จาก hyperparameter ที่ชนะ แล้ว fit ด้วยข้อมูล **train + validation** เพื่อใช้ข้อมูลเรียนรู้ให้มากขึ้น ก่อนประเมิน final score บน test set ชุดเดิม

สำหรับ XGBoost, LightGBM และ CatBoost จะนำจำนวน iteration ที่ดีที่สุดจากรอบ early stopping มาใช้เป็นจำนวน estimator/iteration ของ final model เพื่อไม่ต้องใช้ test set หรือ validation set ซ้ำในการหยุด training


In [ ]:
print("\n" + "=" * 60)
print("5. Training the final selected model on TRAIN + VALIDATION ...")
print("=" * 60)

def best_iteration_from_fitted_model(name, fitted_model):
    if name == "XGBoost":
        best_iteration = getattr(fitted_model, "best_iteration", None)
        return None if best_iteration is None else int(best_iteration) + 1
    if name == "LightGBM":
        best_iteration = getattr(fitted_model, "best_iteration_", None)
        return None if best_iteration is None or best_iteration <= 0 else int(best_iteration)
    if name == "CatBoost":
        best_iteration = fitted_model.get_best_iteration()
        return None if best_iteration is None else int(best_iteration) + 1
    return None

def build_final_model(name, params, tuned_model=None, use_gpu=USE_GPU):
    params = params.copy()
    best_n_estimators = best_iteration_from_fitted_model(name, tuned_model) if tuned_model is not None else None

    if name == "LogisticRegression":
        return LogisticRegression(C=1.0, solver="lbfgs", penalty="l2", max_iter=500, random_state=42)
    if name == "DecisionTree":
        return DecisionTreeClassifier(**params, random_state=42)
    if name == "RandomForest":
        return RandomForestClassifier(**params, bootstrap=True, random_state=42, n_jobs=-1)
    if name == "XGBoost":
        if best_n_estimators is not None:
            params["n_estimators"] = best_n_estimators
        if use_gpu:
            gpu_params = {**gpu_params_for("XGBoost"), "sampling_method": "gradient_based"}
        else:
            gpu_params = {"tree_method": "hist", "sampling_method": "uniform"}
        return xgb.XGBClassifier(**params, eval_metric="logloss", random_state=42, n_jobs=-1, verbosity=0, **gpu_params)
    if name == "LightGBM":
        if best_n_estimators is not None:
            params["n_estimators"] = best_n_estimators
        gpu_params = gpu_params_for("LightGBM") if use_gpu else {}
        return lgb.LGBMClassifier(**params, random_state=42, n_jobs=-1, verbose=-1, **gpu_params)
    if name == "CatBoost":
        if best_n_estimators is not None:
            params["iterations"] = best_n_estimators
        gpu_params = gpu_params_for("CatBoost") if use_gpu else {}
        return CatBoostClassifier(**params, random_seed=42, verbose=0, **gpu_params)
    raise ValueError(f"Unknown model name: {name}")

uses_scaled = best_name == "LogisticRegression"
if uses_scaled:
    X_train_final = np.vstack([X_train_sc, X_val_sc])
else:
    X_train_final = pd.concat([X_train, X_val], axis=0).values

y_train_final = pd.concat([y_train, y_val], axis=0).values
X_test_final = X_test_sc if uses_scaled else X_test.values

final_model = fit_with_optional_gpu_fallback(
    best_name,
    lambda use_gpu: build_final_model(
        best_name,
        results[best_name]["best_params"],
        tuned_model=results[best_name]["model"],
        use_gpu=use_gpu,
    ),
    lambda model: model.fit(X_train_final, y_train_final),
)

final_pred = final_model.predict(X_test_final)
final_prob = final_model.predict_proba(X_test_final)[:, 1]

final_metrics = pd.DataFrame([{
    "Model": best_name,
    "Feature_Set": "Aggregate-Only",
    "Train_Rows": len(y_train_final),
    "Feature_Count": len(agg_feature_cols),
    "Test_Accuracy": accuracy_score(y_test, final_pred),
    "Test_AUC": roc_auc_score(y_test, final_prob),
    "Test_F1": f1_score(y_test, final_pred),
}])

display(final_metrics)
final_metrics.to_csv(OUTPUT_DIR / "final_metrics.csv", index=False)
joblib.dump(final_model, OUTPUT_DIR / "final_best_model.joblib")

numeric_conclusion = (
    f"LogisticRegression Test AUC={summary.loc['LogisticRegression', 'Test_AUC']:.4f}; "
    f"Best Model={best_name}; "
    f"Best Model final Test AUC={final_metrics.loc[0, 'Test_AUC']:.4f}; "
    f"AUC gap={final_metrics.loc[0, 'Test_AUC'] - summary.loc['LogisticRegression', 'Test_AUC']:+.4f}; "
    f"Best Model Accuracy={final_metrics.loc[0, 'Test_Accuracy']:.4f}; "
    f"Best Model F1={final_metrics.loc[0, 'Test_F1']:.4f}."
)

print("\nNumeric Conclusion:")
print(numeric_conclusion)
(OUTPUT_DIR / "conclusion.txt").write_text(numeric_conclusion, encoding="utf-8")

print(f"\nFinal model saved to: {OUTPUT_DIR / 'final_best_model.joblib'}")
print(f"Final metrics saved to: {OUTPUT_DIR / 'final_metrics.csv'}")


## 13. Output Files And Report Notes

หลังรัน notebook ครบ จะได้ไฟล์ใน `outputs/aggregate_only`

- `summary.csv` / `model_numeric_comparison.csv`: ตารางเปรียบเทียบ metric ทุกโมเดลแบบตัวเลข
- `logreg_vs_best_summary.csv` และ `logreg_vs_best_auc.png`: ตาราง/กราฟเทียบ Logistic Regression กับ Best Model
- `best_params.csv`: best hyperparameters ของแต่ละโมเดล
- `final_metrics.csv`: metric ของ final best model ที่ train ใหม่ด้วย train + validation
- `model_comparison.png`: grouped bar chart เปรียบเทียบ AUC / Accuracy / F1 ของทุกโมเดล
- `best_model_eval.png`: ROC curve และ confusion matrix ของ Best Model
- `feature_importance.csv`: feature importance/coefficient ของ Logistic Regression และ Best Model
- `feature_importance_logisticregression.png` และ `feature_importance_<best_model>.png`: top feature importance ของสองโมเดลหลัก
- `final_best_model.joblib`: serialized final model
- `conclusion.txt`: numeric conclusion หลังรัน

Flow สำหรับนำเสนอ: แสดง model comparison ทุกโมเดลแบบตัวเลขและ visualization → เลือก Best Model จาก Test AUC → เทียบ Best Model กับ Logistic Regression → แสดง feature importance ของสองตัวนี้
